# Practica 2 (10-15 min): GraphRAG-light (SOLUCIONES)
## Master Oficial: Big Data Science

Version de referencia para soporte en clase.

In [ ]:
import random
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

import torch
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GATv2Conv

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = Path("../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

K_TOP = 12
MAX_EXPANDED_NODES = 250
MAX_PLOT_NODES = 60

print(f"Device: {device} | Data dir: {DATA_DIR.resolve()}")

In [ ]:
def load_dataset_with_fallback(data_dir: Path):
    last_error = None
    for name in ["Cora", "CiteSeer"]:
        try:
            ds = Planetoid(root=str(data_dir / "planetoid"), name=name)
            return ds, ds[0], name
        except Exception as exc:
            last_error = exc
            print(f"Aviso: no se pudo cargar {name}: {type(exc).__name__}: {exc}")
    raise RuntimeError(f"No se pudo cargar Cora ni CiteSeer: {last_error}")

dataset, data, dataset_name = load_dataset_with_fallback(DATA_DIR)
data = data.to(device)

print(f"Dataset activo: {dataset_name}")
print(f"nodes={data.num_nodes} | edges={data.num_edges} | classes={dataset.num_classes}")

In [ ]:
class QuickGAT(torch.nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, heads=4, dropout=0.6):
        super().__init__()
        self.dropout = dropout
        self.gat1 = GATv2Conv(in_dim, hidden_dim, heads=heads)
        self.gat2 = GATv2Conv(hidden_dim * heads, out_dim, heads=1)

    def forward(self, x, edge_index):
        h = F.dropout(x, p=self.dropout, training=self.training)
        h = self.gat1(h, edge_index)
        h = F.elu(h)
        emb = h
        h = F.dropout(h, p=self.dropout, training=self.training)
        logits = self.gat2(h, edge_index)
        return logits, emb


def quick_train_gat(data, in_dim, out_dim, device, epochs=60):
    model = QuickGAT(in_dim, hidden_dim=16, out_dim=out_dim).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
    criterion = torch.nn.CrossEntropyLoss()

    model.train()
    for _ in range(epochs):
        optimizer.zero_grad()
        logits, _ = model(data.x, data.edge_index)
        loss = criterion(logits[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        logits, embeddings = model(data.x, data.edge_index)

    pred = logits.argmax(dim=1)
    test_acc = float((pred[data.test_mask] == data.y[data.test_mask]).float().mean().item())
    return model, embeddings.detach().cpu(), test_acc


model, embeddings, test_acc = quick_train_gat(data, dataset.num_features, dataset.num_classes, device, epochs=60)
y_cpu = data.y.detach().cpu()
train_idx = torch.where(data.train_mask.detach().cpu())[0]
test_idx = torch.where(data.test_mask.detach().cpu())[0]

print(f"Quick GAT test_acc={test_acc:.4f}")

In [ ]:
# TODO 1 (resuelto): query por centroide de clase
classes, counts = torch.unique(y_cpu[train_idx], return_counts=True)
target_class = int(classes[counts.argmax()].item())
query_vec = embeddings[train_idx[y_cpu[train_idx] == target_class]].mean(dim=0)

print(f"target_class={target_class}")

In [ ]:
# TODO 2 (resuelto): retrieval top-k por coseno sobre nodos de test
sims = F.cosine_similarity(embeddings[test_idx], query_vec.unsqueeze(0), dim=1)
top_local = torch.topk(sims, k=min(K_TOP, sims.numel())).indices
seeds = test_idx[top_local]
seed_scores = sims[top_local]

assert len(seeds) == K_TOP

In [ ]:
# TODO 3 (resuelto): expansion 1-hop
edge_index_cpu = data.edge_index.detach().cpu()
neighbors = defaultdict(set)
for u, v in edge_index_cpu.t().tolist():
    neighbors[u].add(v)
    neighbors[v].add(u)

expanded_set = set(int(n) for n in seeds.tolist())
for s in seeds.tolist():
    expanded_set.update(neighbors[int(s)])

expanded_nodes = sorted(list(expanded_set))[:MAX_EXPANDED_NODES]
assert len(expanded_nodes) >= len(seeds)

In [ ]:
hits_at_k = float((y_cpu[seeds] == target_class).float().mean().item())
purity_expanded = float((y_cpu[expanded_nodes] == target_class).float().mean().item())
expansion_factor = len(expanded_nodes) / len(seeds)

metrics_df = pd.DataFrame([
    {
        "dataset": dataset_name,
        "target_class": int(target_class),
        "hits@k": round(hits_at_k, 4),
        "purity_expanded": round(purity_expanded, 4),
        "expansion_factor": round(expansion_factor, 2),
    }
])
display(metrics_df)
print("Checkpoint metricas OK")

In [ ]:
node_set = list(seeds.tolist())
for n in expanded_nodes:
    if n not in node_set:
        node_set.append(int(n))
    if len(node_set) >= MAX_PLOT_NODES:
        break

G = nx.Graph()
G.add_nodes_from(node_set)
for u in node_set:
    for v in neighbors[u]:
        if v in G:
            G.add_edge(u, v)

score_map = {int(n): 0.0 for n in node_set}
for n, s in zip(seeds.tolist(), seed_scores.tolist()):
    score_map[int(n)] = float(s)

node_colors = [int(y_cpu[n]) for n in G.nodes()]
node_sizes = [220 + 1200 * max(0.0, score_map[int(n)]) for n in G.nodes()]

plt.figure(figsize=(8, 6))
pos = nx.spring_layout(G, seed=42)
nx.draw_networkx_edges(G, pos, alpha=0.25, width=0.8)
nodes = nx.draw_networkx_nodes(
    G,
    pos,
    node_color=node_colors,
    node_size=node_sizes,
    cmap=plt.cm.tab10,
    alpha=0.9,
    edgecolors="black",
    linewidths=0.3,
)
plt.colorbar(nodes, label="class label")
plt.title(f"GraphRAG-light | dataset={dataset_name} | target_class={int(target_class)}")
plt.axis("off")
plt.show()

## Interpretacion sugerida

- Reporta si la expansion incremento o redujo pureza.
- Comenta brevemente si el subgrafo parece coherente con la clase objetivo.